In [9]:
from langchain_core.runnables import RunnableBranch, RunnableParallel, RunnableLambda, RunnableSequence, RunnablePassthrough

In [ ]:
# types of runnables - > task specific runnables, Runnable primitives

# below both are same

# chain =prompt | model | parser. ------> we will use it.
# chain = RunnableSequence(prompt, model , parse)
 

## Runnables Sequence

In [7]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate


prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

model1 = ChatOpenAI(model = "gpt-5.4-nano")
parser = StrOutputParser()

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

chain = RunnableSequence(prompt1, model1, parser, prompt2,model1, parser)

result = chain.invoke({'topic':'AI'})
print(result)

The joke plays on a nerdy pun.

- **“Over neural networks”** sounds like **“overheads”** or **“over”** in a bar context, but it’s actually referencing how AI works (“neural networks”).
- The **ladder** is the “mechanical” part of the punchline: it’s meant to help you **reach the top shelf**.
- So the AI’s reasoning is: *if the good drinks are “over neural networks,” I should use a ladder to reach them.*

It’s basically a mash-up of **AI jargon** with the familiar **bar-top-shelf** setup.


## Runnable passthrough

In [10]:
## runnable passthrough keep the input same in the output

prompt1 = PromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

parser = StrOutputParser()

prompt2 = PromptTemplate(
    template='Explain the following joke - {text}',
    input_variables=['text']
)

joke_gen_chain = RunnableSequence(prompt1, model1, parser)

parallel_chain = RunnableParallel({
    'joke': RunnablePassthrough(),
    'explanation': RunnableSequence(prompt2, model1, parser)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic':'cricket'}))

{'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard it was going to be a *high-stakes* game—so they wanted to be ready to reach the **six**!', 'explanation': 'The joke plays on the word **“six.”**\n\n- In cricket, a **six** is a scoring outcome: if a batsman hits the ball far enough, it’s worth **6 runs**.\n- A **ladder** is something you use to **reach** high places.\n- “**High-stakes**” sounds like something serious and exciting—so the punchline turns that into “high” literally, meaning “high enough to reach a six.”\n\nSo the team brought a ladder because they expected a **high-stakes** match, and they wanted to be literally *ready to reach the six* (i.e., score a big six).'}


## Runnable Lambda

In [14]:
## Runnable Lambda
# it converts a python code to runnable

prompt1 = PromptTemplate(template="generate a detailed report on {topic}" , input_variables=["topic"])

def word_count(text):
    return len(text.split())

summary = prompt1 | model1 | parser

parallel_chain = RunnableParallel({
    "summary": RunnablePassthrough(),
    "words" : RunnableLambda(word_count)
})
chain = summary | parallel_chain

result = chain.invoke({"topic":"AI"})
print(result)


{'summary': '## Detailed Report on Artificial Intelligence (AI)\n\n### 1. Executive Summary\nArtificial Intelligence (AI) refers to computer systems designed to perform tasks that typically require human intelligence, such as perception, reasoning, learning, language understanding, and decision-making. Over the last decade, advances in machine learning—especially deep learning—have accelerated AI adoption across industries, including healthcare, finance, manufacturing, transportation, retail, and entertainment.  \nThis report reviews AI fundamentals, major approaches, key applications, benefits, limitations, risks, ethical considerations, governance frameworks, and future directions.\n\n---\n\n### 2. What Is AI?\nAI is a field of computer science focused on building systems that can:\n- **Perceive** (e.g., interpret images, speech, sensor data)\n- **Learn** from data or interaction\n- **Reason** and make predictions\n- **Understand language** and generate text or audio\n- **Act** to ac

In [15]:
result["words"]

1328

## Runnable Branch

In [ ]:
# work like as if else condition
prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Summarize the following text \n {text}',
    input_variables=['text']
)
report_gen_chain = prompt1 | model1 | parser

branch_chain = RunnableBranch(
    (lambda x: len(x.split())> 500, prompt2|model1|parser),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic':'Russia vs Ukraine'}))


The text summarizes the Russia–Ukraine war as a conflict that escalated from tensions starting in 2014 (Crimea and the Donbas) into a full-scale invasion in February 2022. It outlines the competing stated goals of Russia (security concerns, opposition to NATO influence, protection of Russian speakers, and strategic/geopolitical aims) and Ukraine (defending sovereignty, ending the invasion, reversing occupation, and choosing its own alliances).  

Militarily, it describes an initial phase of attempts at rapid advances followed by a shift in many areas toward positional, attritional trench/line warfare, with heavy use of artillery, drones, electronic warfare, long-range strikes, and improved air defense and logistics. It notes ongoing high-intensity fighting in some regions and lower-intensity patterns elsewhere, alongside persistent missile/drone/cyber activity.  

Human impact is emphasized: millions displaced, extensive damage to civilian infrastructure, significant humanitarian needs